# Device mapping

Put device names in element alias field.

The master mapping of device name to element name is from the SLACPROD Oracle Database, downloaded as a CSV file:

https://oraweb.slac.stanford.edu/apex/slacprod/f?p=116:600


In [1]:
import pandas as pd
import numpy as np
import os

# SLACPROD Oracle Table

In [2]:
# Table extracted from SLACPROD Oracle Database

MASTER = 'from_oracle/lcls_elements.csv'

df = pd.read_csv(MASTER)

# Remove empty
df = df[['Element', 'Control System Name']].dropna()

In [3]:
# These are unique
MADNAMES = list(df['Element'])
len(MADNAMES), len(set(MADNAMES))

(3614, 3614)

In [4]:
# These are not
DEVICENAMES = list(df['Control System Name'])
len(DEVICENAMES), len(set(DEVICENAMES))

(3614, 3201)

In [5]:
# These devices have multiple elements - a mistake?
series  = df.groupby('Control System Name')['Element'].apply(list)
for i, val in series.items():
    if len(val) > 1:
        # Skip klystrons - these are expected to be duplicated
        if not val[0].startswith('K'):
            print(i, val)

In [6]:
# dict for lookup
DEVICE = dict(zip(MADNAMES, DEVICENAMES))

# PyTao

In [7]:
from pytao import Tao
import os

In [8]:
def ele_names(model):
    tao = Tao('-init ../models/'+model+'/tao.init -noplot')
    names = tao.cmd('python lat_list 1@0>>*|model ele.name')
    return names

def remove_superslaves(names):
    return [x for x in names if '#' not in x]

# All models
[d for d in os.listdir('../models') if os.path.isdir('../models/'+d)]


['cu_sxr',
 'hxr',
 'cu_spec',
 'lcls_complex',
 'sc_diag0',
 'cu_hxr',
 'cu_inj',
 'sc_dasel',
 'cu_linac',
 'sc_sxr',
 'sc_hxr']

In [9]:
def write_devicenames(unames, filename):
    my_names = remove_superslaves(unames)
    lines = ['! ---------',
             '! Device mapping derived from '+MASTER
             
            ]
    for name in my_names:
        if name in DEVICE:
            line = name+'[alias]='+ DEVICE[name]
            
        else:
            #continue
            line = '! No device listed for: '+name
        lines.append(line)    
    with open(filename, 'w') as f:
        for line in lines:
            f.write(line+'\n')
    print('Written:', filename)

# Add to CU Master

In [10]:
# Output filename
CU_FILE = '../master/LCLScu_devicenames.bmad'

In [11]:
models = ['cu_hxr', 'cu_sxr', 'cu_spec']
names = []
for m in models:
    print(m)
    names += ele_names(m)
unames = sorted(list(set(names)))

cu_hxr
cu_sxr
cu_spec


In [12]:
write_devicenames(unames, CU_FILE)

Written: ../master/LCLScu_devicenames.bmad


# Add to SC Master

In [13]:
SC_FILE='../master/LCLSsc_devicenames.bmad'

In [14]:
models = ['sc_hxr', 'sc_sxr']
names = []
for m in models:
    print(m)
    names += ele_names(m)
unames = sorted(list(set(names)))

sc_hxr
sc_sxr


In [15]:
write_devicenames(unames, SC_FILE)

Written: ../master/LCLSsc_devicenames.bmad


# elementdevices (old method)

In [16]:
ELEMENTDEVICES = '../../mad/elementdevices.dat'
os.path.exists(ELEMENTDEVICES)

True

In [17]:
def parse_elementdevices(elementdevices_filename):
    """
    
    Parameters
    ----------
    elementdevices_filename
    
    Returns
    -------
    device: dict of ele_name:devicename
    not_found: list of ele_names with no device
    
    """
    device = {}
    not_found = []
    with open(elementdevices_filename) as f:
        for line in f:
            x = line.split()
            if len(x) != 2:
                continue

            ele, devicename = x
            if devicename == '-':
                not_found.append(ele)
                continue
            if ele in device:
                raise ValueError('ele already has a a device:', ele, device[ele])

            device[ele] = devicename   
            
    return device, not_found
DNAME, NOT_FOUND = parse_elementdevices(ELEMENTDEVICES)    
len(list(DNAME)), len(NOT_FOUND)

(2841, 435)

In [18]:
# Check for missing or mismatched items
for ele, dev in DNAME.items():
    if ele not in DEVICE:
        #continue
        print('Missing from Oracle table:', ele, dev)
    else:
        oracle_dev = DEVICE[ele] 
        if oracle_dev != dev:
        #    continue
            print('Device mismatch for ele:', ele, oracle_dev, dev)

Missing from Oracle table: IMBCSI1 TORO:HTR:
Missing from Oracle table: IMBCSI2 TORO:HTR:
Missing from Oracle table: BXG_TRIM BTRM:IN20:231
Missing from Oracle table: BXH1_TRIM BTRM:IN20:451
Missing from Oracle table: BXH3_TRIM BTRM:IN20:475
Missing from Oracle table: BXH4_TRIM BTRM:IN20:481
Missing from Oracle table: BX01_TRIM BTRM:IN20:661
Missing from Oracle table: BX11_TRIM BTRM:LI21:215
Missing from Oracle table: BX13_TRIM BTRM:LI21:241
Missing from Oracle table: BX14_TRIM BTRM:LI21:261
Missing from Oracle table: BX21_TRIM BTRM:LI24:720
Missing from Oracle table: BX23_TRIM BTRM:LI24:810
Missing from Oracle table: BX24_TRIM BTRM:LI24:870
Device mismatch for ele: P30013 LI30:PROF:13 PROF:LI30:13
Device mismatch for ele: P30014 LI30:PROF:14 PROF:LI30:14
Device mismatch for ele: P30143 LI30:PROF:143 PROF:LI30:143
Device mismatch for ele: P30144 LI30:PROF:144 PROF:LI30:144
Device mismatch for ele: P30443 LI30:PROF:443 PROF:LI30:443
Device mismatch for ele: P30444 LI30:PROF:444 PROF:LI3